In [ ]:
# Source - https://stackoverflow.com/a
# Posted by G M, modified by community. See post 'Timeline' for change history
if 'google.colab' in str(get_ipython()):
  !git clone https://github.com/Vladislavicious/jenga_ml.git
  %cd jenga_ml
  !git switch dev

  !pip install -r requirements.txt
else:
  print('Not running on CoLab')

In [ ]:
import random
from environment import make_jenga_env
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env


In [ ]:
n_blocks = 1
random.seed(123)
np.random.seed(123)
CHECK_STEPS = 1024

def make_env():
    return make_jenga_env(n_blocks=n_blocks, render=True)


In [ ]:
num_envs = 8
env = make_vec_env(make_env, n_envs=num_envs, vec_env_cls=SubprocVecEnv)
# env = make_env()

In [ ]:

model = PPO(
    "MlpPolicy",
    env,
    n_steps=CHECK_STEPS,
    batch_size=128,
    verbose=1,
    seed=123,
)

model.learn(total_timesteps=300000)

In [ ]:
model.save("jenga_ppo_singlethread")

In [ ]:
single_env = make_env()

In [ ]:
obs, _ = single_env.reset()
for i in range(CHECK_STEPS * 2):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = single_env.step(action)
    single_env.render()
    if terminated or truncated:
        obs, _ = single_env.reset()
    single_env.env.debug_output()
    if i == CHECK_STEPS - 2:
        print("hi")

In [ ]:
env.env.debug_output()